In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%reload_ext autoreload

In [0]:
from pyspark.sql.functions import col,max
from scr.utils.config import catalog_name,schema_name,bronze_table,checkpoint_dim_location,location_table
from src.transformations.silver_transformations import merge_dim_location

In [0]:
bronze_stream = (
    spark.readStream
         .option("readChangeFeed", "true")
         .table(f"{catalog_name}.{schema_name}.{bronze_table}")
         .filter(col("_change_type").isin("insert","update_postimage"))
         )

query=(
bronze_stream.writeStream
    .foreachBatch(merge_dim_location)
    .option("checkpointLocation",checkpoint_dim_location)
    .trigger(once=True)
    .start()
)
query.awaitTermination()